# Lab 1 — Experiment No. 1
## Deterministic Tutoring Agent — Version 2

**Progression:** Basic → Optimization 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10

This version is designed so that every optimization has a clear **BEFORE / AFTER** demonstration suitable for screenshots and viva explanation.

## 1. Install and Import Libraries

In [ ]:
!pip -q install -U google-genai pandas

In [ ]:
import json
import sqlite3
import time
from datetime import datetime
import pandas as pd
from google import genai

API_KEY = "YOUR_GEMINI_API_KEY"
client = genai.Client(api_key=API_KEY)

# If the model is unavailable for your key, run:
# [m.name for m in client.models.list()]
MODEL = "gemini-3.5-flash-lite"

print("Gemini client initialized.")

## 2. Knowledge Base

In [ ]:
knowledge_base = {
    "stack": {
        "keywords": ["stack", "push", "pop", "lifo"],
        "response": {
            "Topic": "Stack",
            "Definition": "A Stack is a linear data structure that follows the LIFO (Last In, First Out) principle.",
            "Algorithm": "Push inserts an element at the top and Pop removes the top element.",
            "Complexity": "Push: O(1), Pop: O(1)",
            "Example": "If A, B and C are pushed into a stack, C is removed first.",
            "Summary": "A Stack follows the LIFO principle."
        }
    },
    "queue": {
        "keywords": ["queue", "enqueue", "dequeue", "fifo"],
        "response": {
            "Topic": "Queue",
            "Definition": "A Queue is a linear data structure that follows the FIFO (First In, First Out) principle.",
            "Algorithm": "Enqueue inserts an element at the rear and Dequeue removes an element from the front.",
            "Complexity": "Enqueue: O(1), Dequeue: O(1)",
            "Example": "If A, B and C are inserted into a queue, A is removed first.",
            "Summary": "A Queue follows the FIFO principle."
        }
    },
    "binary search": {
        "keywords": ["binary search"],
        "response": {
            "Topic": "Binary Search",
            "Definition": "Binary Search is a searching algorithm that works on a sorted array.",
            "Algorithm": "Compare the target with the middle element and repeatedly search the appropriate half.",
            "Complexity": "Time Complexity: O(log n)",
            "Example": "A sorted array can be searched by repeatedly dividing the search interval into two halves.",
            "Summary": "Binary Search efficiently searches sorted data by repeatedly halving the search space."
        }
    }
}

with open("templates.json", "w") as f:
    json.dump(knowledge_base, f, indent=4)

FALLBACK = "Sorry, I can answer only predefined academic questions."

def detect_topic(question):
    q = question.lower()
    for topic, data in knowledge_base.items():
        for keyword in data["keywords"]:
            if keyword in q:
                return topic
    return None

def get_template(topic):
    return knowledge_base[topic]["response"]

# BASIC AGENT

### Baseline
Only keyword matching and predefined template retrieval are used.

In [ ]:
def basic_agent(question):
    topic = detect_topic(question)
    if not topic:
        return None, FALLBACK
    return topic, get_template(topic)

question = "What is a stack?"
topic, response = basic_agent(question)

print("=" * 70)
print("BASIC AGENT — BASELINE")
print("=" * 70)
print("Question:", question)
print("Detected Topic:", topic)
print("\nOutput:")
print(json.dumps(response, indent=2))

# OPTIMIZATION 1 — Prompt Optimization

**BEFORE:** Generic instruction.

**AFTER:** Strict instruction with fixed formatting and no extra information.

In [ ]:
PROMPT_BEFORE = "Answer the question using the template."

PROMPT_AFTER = """
You are a university tutoring assistant.
Use ONLY the supplied template.
Do not add information.
Do not paraphrase.
Return exactly these sections:
Topic, Definition, Algorithm, Complexity, Example, Summary.
"""

print("=" * 70)
print("OPTIMIZATION 1 — PROMPT OPTIMIZATION")
print("=" * 70)
print("\nBEFORE PROMPT:\n", PROMPT_BEFORE)
print("\nAFTER PROMPT:\n", PROMPT_AFTER)
print("\nImprovement: Clear restrictions + fixed response structure.")

# OPTIMIZATION 2 — Intent Classification

**BEFORE:** Topic only.

**AFTER:** Topic + student's learning objective.

In [ ]:
def classify_intent(question):
    q = question.lower()

    if any(x in q for x in ["debug", "error", "fix"]):
        return "Debugging"
    if any(x in q for x in ["write code", "code for", "implement"]):
        return "Code Generation"
    if any(x in q for x in ["compare", "difference between"]):
        return "Comparison"
    if "example" in q:
        return "Example"
    if any(x in q for x in ["complexity", "big o"]):
        return "Complexity Analysis"
    if any(x in q for x in ["algorithm", "steps", "how does it work"]):
        return "Algorithm"
    if any(x in q for x in ["what is", "define", "definition", "explain"]):
        return "Definition"
    return "General"

question = "What is the complexity of Binary Search?"

print("=" * 70)
print("OPTIMIZATION 2 — INTENT CLASSIFICATION")
print("=" * 70)
print("\nBEFORE:")
print("Topic:", detect_topic(question))
print("Intent: Not detected")

print("\nAFTER:")
print("Topic:", detect_topic(question))
print("Intent:", classify_intent(question))
print("\nImprovement: The agent now knows what the student wants to learn.")

# OPTIMIZATION 3 — Dynamic Prompt Selection

**BEFORE:** One generic prompt.

**AFTER:** Prompt selected according to learning intent.

In [ ]:
PROMPTS = {
    "Definition": "Explain the definition clearly using only the template.",
    "Algorithm": "Focus on the algorithm and steps using only the template.",
    "Complexity Analysis": "Focus on complexity using only the template.",
    "Example": "Focus on the example using only the template.",
    "Comparison": "Focus on comparison using only supplied information.",
    "Code Generation": "Focus on programming information using only the template.",
    "Debugging": "Focus on debugging information using only the template.",
    "General": "Explain using only the supplied template."
}

question = "What is the complexity of Binary Search?"
intent = classify_intent(question)

print("=" * 70)
print("OPTIMIZATION 3 — DYNAMIC PROMPT SELECTION")
print("=" * 70)
print("\nBEFORE: One generic tutoring prompt")
print("\nAFTER:")
print("Detected Intent:", intent)
print("Selected Prompt:", PROMPTS[intent])
print("\nImprovement: Different learning tasks receive different instructions.")

# OPTIMIZATION 4 — Clarification Agent

**BEFORE:** Ambiguous question is answered immediately.

**AFTER:** Agent asks the student to specify the intended topic.

In [ ]:
AMBIGUOUS = {
    "tree": ["Binary Tree", "AVL Tree", "B-Tree", "Decision Tree"],
    "trees": ["Binary Tree", "AVL Tree", "B-Tree", "Decision Tree"],
    "search": ["Linear Search", "Binary Search"]
}

def clarification(question):
    q = question.lower().strip()
    for word, choices in AMBIGUOUS.items():
        if q in [word, "explain " + word, "what is " + word]:
            return choices
    return None

question = "Explain Trees"
choices = clarification(question)

print("=" * 70)
print("OPTIMIZATION 4 — CLARIFICATION AGENT")
print("=" * 70)
print("\nBEFORE:")
print("Question:", question)
print("Action: Attempt to answer immediately")

print("\nAFTER:")
print("Question:", question)
print("Agent: Your question is ambiguous. Do you mean:")
for i, choice in enumerate(choices, 1):
    print(f"{i}. {choice}")
print("\nImprovement: Ambiguous queries are clarified instead of guessed.")

# OPTIMIZATION 5 — Response Validation

**BEFORE:** Response is returned without completeness checking.

**AFTER:** Required sections are checked automatically.

In [ ]:
REQUIRED = ["Topic", "Definition", "Algorithm", "Complexity", "Example", "Summary"]

def validate_response(text):
    missing = [x for x in REQUIRED if x.lower() not in text.lower()]
    return len(missing) == 0, missing

bad_response = """
Topic: Stack
Definition: A Stack follows LIFO.
Complexity: Push and Pop are O(1).
"""

good_response = """
Topic: Stack
Definition: A Stack is a linear data structure that follows LIFO.
Algorithm: Push inserts at the top and Pop removes from the top.
Complexity: Push: O(1), Pop: O(1)
Example: A, B, C are pushed and C is removed first.
Summary: A Stack follows LIFO.
"""

print("=" * 70)
print("OPTIMIZATION 5 — RESPONSE VALIDATION")
print("=" * 70)

ok, missing = validate_response(bad_response)
print("\nBEFORE:")
print("Validation:", "PASS" if ok else "FAIL")
print("Missing:", missing)

ok, missing = validate_response(good_response)
print("\nAFTER:")
print("Validation:", "PASS" if ok else "FAIL")
print("Missing:", missing if missing else "None")
print("\nImprovement: Incomplete responses are detected automatically.")

# OPTIMIZATION 6 — Self-Reflection

**BEFORE:** Incomplete response is returned.

**AFTER:** The agent identifies missing sections and repairs them from the existing template.

In [ ]:
def self_reflect(draft, correct_template):
    draft = dict(draft)
    missing = [x for x in REQUIRED if x not in draft]
    for key in missing:
        draft[key] = correct_template[key]
    return draft

draft = {
    "Topic": "Stack",
    "Definition": "A Stack is a linear data structure that follows LIFO.",
    "Complexity": "Push: O(1), Pop: O(1)"
}

improved = self_reflect(draft, get_template("stack"))

print("=" * 70)
print("OPTIMIZATION 6 — SELF-REFLECTION")
print("=" * 70)
print("\nBEFORE:")
print(json.dumps(draft, indent=2))
print("Missing:", [x for x in REQUIRED if x not in draft])

print("\nAFTER:")
print(json.dumps(improved, indent=2))
print("Missing:", [x for x in REQUIRED if x not in improved] or "None")
print("\nImprovement: Missing sections are repaired before returning the answer.")

# OPTIMIZATION 7 — Retry Strategy

**BEFORE:** First failed response is accepted.

**AFTER:** Failed validation triggers a stricter retry, up to two retries.

In [ ]:
def simulated_model_response(attempt):
    if attempt == 1:
        return """
Topic: Stack
Definition: A Stack follows LIFO.
Complexity: Push and Pop are O(1).
"""
    return """
Topic: Stack
Definition: A Stack follows LIFO.
Algorithm: Push inserts at the top and Pop removes the top element.
Complexity: Push: O(1), Pop: O(1)
Example: If A, B and C are pushed, C is removed first.
Summary: A Stack follows LIFO.
"""

print("=" * 70)
print("OPTIMIZATION 7 — RETRY STRATEGY")
print("=" * 70)

for attempt in range(1, 4):
    result = simulated_model_response(attempt)
    valid, missing = validate_response(result)

    print(f"\nAttempt {attempt}:")
    print("Validation:", "PASS" if valid else "FAIL")
    print("Missing:", missing if missing else "None")

    if valid:
        print("→ Response accepted.")
        break
    print("→ Retry with stricter prompt.")

print("\nImprovement: Failed output is not immediately accepted.")

# OPTIMIZATION 8 — Adaptive Tutoring

**BEFORE:** Same explanation for every student.

**AFTER:** Beginner, Intermediate, and Advanced explanations are selected.

In [ ]:
adaptive_templates = {
    "Beginner": "A Stack is a data structure where the last item added is the first item removed.",
    "Intermediate": "A Stack is a linear data structure that follows the LIFO (Last In, First Out) principle.",
    "Advanced": "A Stack follows LIFO semantics, with insertion and deletion performed at the top."
}

print("=" * 70)
print("OPTIMIZATION 8 — ADAPTIVE TUTORING")
print("=" * 70)
print("\nBEFORE:")
print(adaptive_templates["Intermediate"])

print("\nAFTER:")
for level, answer in adaptive_templates.items():
    print(f"\n{level}:")
    print(answer)

print("\nImprovement: The explanation can match the student's learning level.")

# OPTIMIZATION 9 — Context Awareness

**BEFORE:** Every question is processed independently.

**AFTER:** The previous question is stored and used to interpret a follow-up.

In [ ]:
conversation_history = []

def context_agent(question):
    conversation_history.append(question)
    conversation_history[:] = conversation_history[-2:]

    if "compare it with stack" in question.lower() and len(conversation_history) >= 2:
        return {
            "Topic": "Queue vs Stack",
            "Context Used": conversation_history[-2],
            "Comparison": "Queue follows FIFO, while Stack follows LIFO.",
            "Summary": "The two data structures differ in the order in which elements are removed."
        }

    return {"Topic": "Queue", "Response": "A Queue follows the FIFO principle."}

conversation_history = []

first = context_agent("Explain Queue")
second = context_agent("Compare it with Stack")

print("=" * 70)
print("OPTIMIZATION 9 — CONTEXT AWARENESS")
print("=" * 70)

print("\nBEFORE:")
print("Question 1: Explain Queue")
print("Question 2: Compare it with Stack")
print("Without context: 'it' has no stored reference.")

print("\nAFTER:")
print("Stored History:", conversation_history)
print("Follow-up Output:")
print(json.dumps(second, indent=2))
print("\nImprovement: The follow-up uses the previous Queue context.")

# OPTIMIZATION 10 — Response Refinement

**BEFORE:** Repeated information and inconsistent formatting.

**AFTER:** Duplicate lines are removed and the response is cleaned.

In [ ]:
messy_response = """
Topic: Stack

Definition: A Stack follows LIFO.

Definition: A Stack follows LIFO.

Complexity: Push: O(1), Pop: O(1).

Example: A, B, C are pushed and C is removed first.

Example: A, B, C are pushed and C is removed first.
"""

def refine_response(text):
    seen = set()
    cleaned = []

    for line in text.splitlines():
        line = line.strip()
        if not line or line in seen:
            continue
        seen.add(line)
        cleaned.append(line)

    return "\n".join(cleaned)

print("=" * 70)
print("OPTIMIZATION 10 — RESPONSE REFINEMENT")
print("=" * 70)
print("\nBEFORE:")
print(messy_response)

print("\nAFTER:")
print(refine_response(messy_response))

print("\nImprovement: Repetition is removed and readability is improved.")

# FINAL AGENT + SQLITE LOGGING

The final section demonstrates a compact deterministic pipeline and records interactions in SQLite.

In [ ]:
conn = sqlite3.connect("database.db")

conn.execute("""
CREATE TABLE IF NOT EXISTS interactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    question TEXT,
    intent TEXT,
    response TEXT,
    response_time REAL
)
""")
conn.commit()

def log_interaction(question, intent, response, response_time):
    conn.execute("""
    INSERT INTO interactions
    (timestamp, question, intent, response, response_time)
    VALUES (?, ?, ?, ?, ?)
    """, (datetime.now().isoformat(), question, intent, response, response_time))
    conn.commit()

def final_tutor(question):
    start = time.time()
    topic = detect_topic(question)

    if not topic:
        return None, FALLBACK, time.time() - start

    response = get_template(topic)
    text = json.dumps(response, indent=2)

    return topic, text, time.time() - start

questions = [
    "What is a stack?",
    "What is the complexity of binary search?",
    "Give an example of a queue"
]

for question in questions:
    topic, response, response_time = final_tutor(question)
    intent = classify_intent(question)
    log_interaction(question, intent, response, response_time)

print("=" * 70)
print("FINAL AGENT — INTERACTION LOG")
print("=" * 70)
print("Interactions logged successfully.")

df = pd.read_sql_query(
    "SELECT * FROM interactions ORDER BY id DESC LIMIT 10",
    conn
)
df

# Result Analysis

| Stage | Visible improvement |
|---|---|
| Basic | Keyword-based template retrieval |
| O1 | Strict prompt and fixed format |
| O2 | Learning objective detection |
| O3 | Task-specific prompt selection |
| O4 | Ambiguous query clarification |
| O5 | Automatic completeness validation |
| O6 | Missing information repaired |
| O7 | Failed response retried |
| O8 | Beginner / Intermediate / Advanced adaptation |
| O9 | Follow-up question uses conversation context |
| O10 | Repeated information removed and formatting improved |

## Expected Outcome

The tutoring agent becomes progressively more structured, robust, adaptive, context-aware, and readable while continuing to use the predefined knowledge base and the permitted Python/LLM workflow.

## Conclusion

The deterministic tutoring agent was implemented and progressively optimized through all ten optimization stages. Each optimization was demonstrated using a BEFORE/AFTER comparison, making the effect of the optimization observable and suitable for experimental analysis.